# 03 — XGBoost Classifier: Chega Growth in Municipal Elections (2021→2022)
Predicts whether Chega's vote share grew between the 2021 municipal and 2022 legislative elections. Uses Leave-One-Out cross-validation and SHAP for feature importance.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report
import shap
from xgboost import XGBClassifier
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv('tabela_principal_autarquicas_2025_v0.csv', sep=';')

In [ ]:
df.columns

In [ ]:
print(df.columns.tolist())

In [ ]:
df["crescimento_chega_bin"] = (df["CH_2022"] > df["CH_2021"]).astype(int)

In [ ]:
# target
y = df["crescimento_chega_bin"]

# features: remover colunas que não devem entrar
X = df.drop(columns=[
    "Concelho",           # identificador
    "CH_2021",            # votes passados do Chega → não pode entrar
    "CH_2022",            # target já definido
    "crescimento_chega_bin" # target
])


In [ ]:
X_clean = X.copy()
X_clean.columns = [c.replace("[","_").replace("]","_").replace(":","_").replace(" ","_") for c in X_clean.columns]


In [ ]:
loo = LeaveOneOut()
y_true, y_pred = [], []

# 7️⃣ Loop Leave-One-Out
for train_idx, test_idx in loo.split(X_clean):
    X_train, X_test = X_clean.iloc[train_idx], X_clean.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    model = XGBClassifier(
        n_estimators=200,
        learning_rate=0.1,
        max_depth=5,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        use_label_encoder=False,
        eval_metric="logloss"
    )

    model.fit(X_train, y_train)
    y_hat = model.predict(X_test)

    y_true.append(y_test.values[0])
    y_pred.append(y_hat[0])


In [ ]:
print("Accuracy LOO:", accuracy_score(y_true, y_pred))
print(classification_report(y_true, y_pred))

In [ ]:
model_final = XGBClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    use_label_encoder=False,
    eval_metric="logloss"
)
model_final.fit(X_clean, y)

In [ ]:
plt.figure(figsize=(12,8))
plt.barh(X_clean.columns, model_final.feature_importances_)
plt.title("Importância das Features - Crescimento Chega")
plt.show()

In [ ]:
# Criar DataFrame com feature names e suas importâncias
feature_importance = pd.DataFrame({
    "Feature": X_clean.columns,
    "Importance": model_final.feature_importances_
})

# Ordenar pela importância (decrescente)
feature_importance = feature_importance.sort_values(by="Importance", ascending=False)

# Mostrar todas as features com importância > 0
print("Features relevantes:")
print(feature_importance[feature_importance["Importance"] > 0])

# Mostrar features com importância zero ou muito baixa
print("\nFeatures pouco relevantes ou irrelevantes:")
print(feature_importance[feature_importance["Importance"] < 0.001])


In [ ]:
# 1️⃣ Filtrar features relevantes
threshold = 0.001
relevant_features = feature_importance[feature_importance["Importance"] >= threshold]["Feature"].tolist()

X_relevant = X_clean[relevant_features]

print(f"Número de features usadas: {len(relevant_features)}")

# 2️⃣ Inicializar Leave-One-Out
loo = LeaveOneOut()
y_true, y_pred = [], []

# 3️⃣ Loop Leave-One-Out com features filtradas
for train_idx, test_idx in loo.split(X_relevant):
    X_train, X_test = X_relevant.iloc[train_idx], X_relevant.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    model = XGBClassifier(
        n_estimators=200,
        learning_rate=0.1,
        max_depth=5,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        use_label_encoder=False,
        eval_metric="logloss"
    )

    model.fit(X_train, y_train)
    y_hat = model.predict(X_test)

    y_true.append(y_test.values[0])
    y_pred.append(y_hat[0])

# 4️⃣ Avaliar performance
from sklearn.metrics import accuracy_score, classification_report

print("Accuracy LOO com features relevantes:", accuracy_score(y_true, y_pred))
print(classification_report(y_true, y_pred))
